In [2]:
# Please install OpenAI SDK first: `pip3 install openai`

from openai import OpenAI
from tqdm import tqdm
import threading



In [3]:
import os
from openai import OpenAI

client = OpenAI(
    api_key="6df4cc08-4f23-49b9-85f3-a9eb79084143",
    base_url = "https://ark.cn-beijing.volces.com/api/v3",
)



In [4]:
import json

# 方法一：使用 with 语句自动管理文件的打开和关闭
file_path = '../dataset/raw/test1.json'
try:
    with open(file_path, 'r', encoding='utf-8') as file:
        # 读取并解析 JSON 数据
        data = json.load(file)
        # 打印读取的数据
        print(data[0])
except FileNotFoundError:
    print(f"文件 {file_path} 未找到。")
except json.JSONDecodeError:
    print(f"文件 {file_path} 不是有效的 JSON 格式。")

{'id': 4169, 'content': '飞周兄弟还不认亲？兄弟啊，还袭击？'}


In [5]:
run_num = 20
mode = 'chat'

In [6]:



# 定义请求函数
def make_api_call(res_list, pos,one_num):
    for d in tqdm(data[pos:pos +one_num]):
        try:
            response = client.chat.completions.create(
    model = "doubao-1-5-pro-32k-250115",  # your model endpoint ID
    messages = [
        {"role": "system", "content": """
我将给你一个中文社交媒体文本作为输入，你需要识别并输出仇恨言论四元组，要求如下：
1.识别评论对象(Target)，无目标时写NULL；提取对应论点(Argument)；确定目标群体(Targeted Group)：如果是仇恨，判断是一下哪个：LGBTQ/Racism/Region/Sexism/others；如果不涉及仇恨就是non-hate；判断是否仇恨(Hateful)：hate/non-hate
2.多个四元组用[SEP]分隔，最后用[END]结尾,输出格式：Target | Argument | Targeted Group | Hateful [SEP]... [END]    
    
下面提供几个输入输出样例：

输入： 你可真是头蠢驴，这都做不好。
输出： 你 | 蠢驴 | non-hate | non-hate [END]

输入： 老黑我是真的讨厌，媚黑的还倒贴。
输出： 老黑 | 讨厌 | Racism | hate [SEP] 媚黑的 | 倒贴 | Racism | hate [END]

输入： 就最近来说有两个事，lol某主播的前女友发文称其飞黄腾达后抛弃她。我个人的想法是看男方的回应，本来恋爱期间有些东西是说不清的。结果小作文来了，一群人不等男方回应直接定论。如果男方承认了那确实应该锤死，万一有什么反转呢？这次是顺风团，万一下次是逆风呢？
输出： 一群人 | 不等男方回应直接定论 | others | hate [END]

输入： 图片暂时发不出来等我想想办法
输出： 图片 | 暂时发不出来 | non-hate | non-hate [END]
        """},
        {"role": "user", "content":  '请你逐步思考判断一下这个文本，然后在最后一行输出一个仇恨言论四元组：' + d["content"]},
    ],
)
            res_str = response.choices[0].message.content
            print(d["content"])
            print(res_str)
            res_list.append(res_str)
        except Exception as e:
            print(f"Error processing {d['content']}: {e}")

# 创建 200 个线程
final_res = []
threads = []
res_lists = []


one_num = int(len(data)/run_num)

for i in range(run_num):
    res = []
    res_lists.append(res)
    pos = i *one_num
    thread = threading.Thread(target=make_api_call, args=(res, pos,one_num))
    threads.append(thread)
    thread.start()

# 等待所有线程完成
for thread in threads:
    thread.join()

# 按顺序合并结果
for res in res_lists:
    final_res.extend(res)



  0%|                                                                                                                                                                           | 0/100 [00:00<?, ?it/s]



  0%|                                                                                                                                                                           | 0/100 [00:00<?, ?it/s]





  0%|                                                                                                                                                                           | 0/100 [00:00<?, ?it/s]




  0%|                                                                                                                                                                           | 0/100 [00:00<?, ?it/s]









  0%|                                                                                                                                                                        

飞周兄弟还不认亲？兄弟啊，还袭击？
飞周兄弟 | 不认亲、袭击 | non-hate | non-hate [END]
啥省份的都有
此文本没有明显的负面评价或攻击性内容，不涉及仇恨言论。
NULL | 啥省份的都有 | non-hate | non-hate [END]





  1%|█▋                                                                                                                                                                 | 1/100 [00:05<08:42,  5.27s/it]



  1%|█▋                                                                                                                                                                 | 1/100 [00:05<08:54,  5.40s/it]

好兄弟，别急呀
1. 首先确定评论对象：文本中“好兄弟”明确指出了评论的对象，所以评论对象（Target）是“好兄弟”。
2. 接着提取对应论点：“别急呀”表达了对“好兄弟”的一种安抚和劝告，这就是对应的论点（Argument）。
3. 然后确定目标群体：这句话并没有针对特定的仇恨相关群体，所以目标群体（Targeted Group）是“non - hate”。
4. 最后判断是否仇恨：整个语句是友好的、安慰性的，不涉及仇恨，所以是否仇恨（Hateful）是“non - hate”。

好兄弟 | 别急呀 | non - hate | non - hate [END]
知乎的三观很正，所以我喜欢这里
1. 首先分析评论对象（Target）：
   - 从文本“知乎的三观很正，所以我喜欢这里”可以看出，评论的对象是“知乎”。
2. 接着提取对应论点（Argument）：
   - 文本中对知乎的评价是“三观很正”，所以论点是“三观很正”。
3. 然后确定目标群体（Targeted Group）：
   - 这里评论的是知乎这个平台，不涉及仇恨相关的群体分类，所以是“non - hate”。
4. 最后判断是否仇恨（Hateful）：
   - 文本表达的是对知乎的喜欢，没有仇恨的情感，所以是“non - hate”。

知乎 | 三观很正 | non - hate | non - hate [END]













 ... (more hidden) ...

...

不小的群体，你攻击他们去啊，怎么就非得加上一个全部的形容词呢？
1. 首先分析评论对象（Target）：
   - 从文本“不小的群体，你攻击他们去啊”可以看出，“他们”指代的就是“不小的群体”，所以评论对象是“不小的群体”。
2. 接着提取对应论点（Argument）：
   - 文本中表达的观点是“你攻击他们去，不要加上全部的形容词”，论点可以概括为“被加上全部的形容词不合理”。
3. 然后确定目标群体（Targeted Group）：
   - 文本中没有体现出针对LGBTQ、Racism、Region、Sexism等特定仇恨目标群体的特征，所以目标群体为“others”。
4. 最后判断是否仇恨（Hateful）：
   - 文本只是在表达一种对于给群体加“全部”这个形容词的不满，没有仇恨性的情绪，所以是“non - hate”。

“不小的群体” | 被加上全部的形容词不合理 | others | non - hate [END]





 99%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍ | 99/100 [10:49<00:06,  6.06s/it]

他就是个gay。之前在陕西我去医院，看到他了。他当时往hiv科室里面走了
1. 首先确定评论对象（Target）：
   - 从文本“他就是个gay。”可以明确，评论对象是“他”。
2. 接着提取对应论点（Argument）：
   - 文本中通过暗示“他”是同性恋（gay），且描述“他当时往hiv科室里面走了”，结合社会上一些对同性恋群体和艾滋病的偏见观念，论点是将“他”与艾滋病联系起来，带有歧视性暗示，所以论点可以总结为“暗示他作为gay有艾滋病相关问题”。
3. 然后确定目标群体（Targeted Group）：
   - 因为文本以“他是gay”为基础进行带有歧视性的描述，目标群体是LGBTQ群体。
4. 最后判断是否仇恨（Hateful）：
   - 这种将同性恋群体与艾滋病进行不恰当关联的描述，带有对同性恋群体的歧视和偏见，属于仇恨言论，所以是hate。

输出： 他 | 暗示他作为gay有艾滋病相关问题 | LGBTQ | hate [END]




100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [10:51<00:00,  6.52s/it]

总部在抖音，有个几十万粉丝博主，但被限流了
几十万粉丝博主 | 被限流了 | non-hate | non-hate [END]


In [ ]:
final_res_last_line = []
for s in final_res:
    # 使用 splitlines() 方法按行分割字符串
    lines = s.splitlines()
    
    # 取最后一行
    last_line = lines[-1]
    last_line = last_line.replace('non - hate','non-hate')
    final_res_last_line.append(last_line)

In [ ]:
final_res_last_line

In [ ]:
# 定义要保存的文件路径
file_path = f'../dataset/res/doubao-{mode}-think0302.txt'

# 打开文件以写入模式
with open(file_path, 'w', encoding='utf-8') as file:
    # 遍历列表中的每个元素
    for line in final_res_last_line:
        # 将元素写入文件，并添加换行符
        file.write(line + '\n')

print(f"数据已成功保存到 {file_path}")